# HyDE and Parent Document Retrieval

You will learn two methods that improve retrieval quality.

You will improve query representation and returned context.


In [ ]:
%pip install -q openai numpy --upgrade

In [1]:
import getpass
import os
import json
import numpy as np
from dataclasses import dataclass, field
from openai import OpenAI

# Set up your OpenAI client
if not os.getenv("OPENAI_API_KEY"):
    secret_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = secret_key

client = OpenAI()
MODEL = "gpt-5-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

## Part 1 HyDE

You will generate a hypothetical answer and embed it for retrieval.


In [2]:
def get_embedding(text: str) -> np.ndarray:
    """Get embedding for a single text."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )
    return np.array(response.data[0].embedding)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def generate_hypothetical_answer(query: str) -> str:
    """Generate a hypothetical document that would answer the query."""
    response = client.responses.create(
        model=MODEL,
        instructions="""Generate a hypothetical document passage that would answer the given question.
The passage should be written as if it's from an authoritative source like a textbook or documentation.
It doesn't need to be perfectly accurate - the goal is to match the style and vocabulary of real documents.
Keep it to 2-3 sentences.""",
        input=query
    )
    return response.output_text

# Demonstrate HyDE
query = "How does TCP handle packet loss?"
hypothetical = generate_hypothetical_answer(query)

print(f"Query: {query}")
print(f"\nHypothetical Answer: {hypothetical}")

Query: How does TCP handle packet loss?

Hypothetical Answer: TCP treats packet loss as a signal to retransmit missing data and to throttle sending rate: the sender detects loss either when a retransmission timeout (RTO) expires or when it receives three duplicate ACKs (fast retransmit), then retransmits the missing segment using sequence numbers and cumulative (or optional SACK) acknowledgments to ensure in-order delivery.  Loss also triggers congestion-control responses—slow start, congestion avoidance, fast retransmit/fast recovery and multiplicative decrease of the congestion window (cwnd)—so TCP both recovers the lost data and reduces sending rate to probe for the available network capacity.


In [3]:
# Sample documents (simulating a knowledge base)
documents = [
    "TCP uses a sliding window protocol with acknowledgments. When packets are lost, "
    "the sender detects this through timeout or duplicate ACKs and retransmits the missing segments.",
    
    "UDP is a connectionless protocol that does not guarantee delivery. "
    "It's faster than TCP but provides no built-in mechanism for handling lost packets.",
    
    "The HTTP protocol operates at the application layer and typically runs over TCP. "
    "HTTP/2 introduced multiplexing to improve performance over single connections.",
    
    "Network congestion occurs when too many packets are sent into the network. "
    "TCP's congestion control algorithms like CUBIC adjust the sending rate dynamically.",
    
    "The three-way handshake (SYN, SYN-ACK, ACK) establishes a TCP connection. "
    "This ensures both parties are ready to communicate before data transfer begins."
]

# Embed all documents
doc_embeddings = [get_embedding(doc) for doc in documents]

def retrieve(query_embedding: np.ndarray, top_k: int = 3) -> list[tuple[str, float]]:
    """Retrieve top-k documents by cosine similarity."""
    similarities = [cosine_similarity(query_embedding, doc_emb) for doc_emb in doc_embeddings]
    ranked = sorted(zip(documents, similarities), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

# Compare standard vs HyDE retrieval
print("=== Standard Retrieval (embed query directly) ===")
query_embedding = get_embedding(query)
standard_results = retrieve(query_embedding)
for doc, score in standard_results:
    print(f"[{score:.3f}] {doc[:80]}...")

print("\n=== HyDE Retrieval (embed hypothetical answer) ===")
hyde_embedding = get_embedding(hypothetical)
hyde_results = retrieve(hyde_embedding)
for doc, score in hyde_results:
    print(f"[{score:.3f}] {doc[:80]}...")

=== Standard Retrieval (embed query directly) ===
[0.641] TCP uses a sliding window protocol with acknowledgments. When packets are lost, ...
[0.543] UDP is a connectionless protocol that does not guarantee delivery. It's faster t...
[0.495] Network congestion occurs when too many packets are sent into the network. TCP's...

=== HyDE Retrieval (embed hypothetical answer) ===
[0.733] TCP uses a sliding window protocol with acknowledgments. When packets are lost, ...
[0.581] Network congestion occurs when too many packets are sent into the network. TCP's...
[0.491] UDP is a connectionless protocol that does not guarantee delivery. It's faster t...


## When HyDE Helps

You should use HyDE when users ask questions and documents store answer style text.


In [4]:
@dataclass
class HyDERetriever:
    """Retriever using Hypothetical Document Embeddings."""
    documents: list[str]
    doc_embeddings: list[np.ndarray] = field(default_factory=list)
    
    def __post_init__(self):
        """Embed documents on initialization."""
        if not self.doc_embeddings:
            print(f"Embedding {len(self.documents)} documents...")
            self.doc_embeddings = [get_embedding(doc) for doc in self.documents]
    
    def retrieve(
        self, 
        query: str, 
        top_k: int = 3, 
        use_hyde: bool = True
    ) -> list[dict]:
        """Retrieve documents, optionally using HyDE."""
        
        if use_hyde:
            # Generate hypothetical answer
            hypothetical = generate_hypothetical_answer(query)
            query_embedding = get_embedding(hypothetical)
            method = "hyde"
        else:
            query_embedding = get_embedding(query)
            hypothetical = None
            method = "standard"
        
        # Compute similarities
        similarities = [
            cosine_similarity(query_embedding, doc_emb) 
            for doc_emb in self.doc_embeddings
        ]
        
        # Rank and return
        ranked_indices = np.argsort(similarities)[::-1][:top_k]
        
        return {
            "method": method,
            "hypothetical": hypothetical,
            "results": [
                {
                    "document": self.documents[i],
                    "score": similarities[i],
                    "rank": rank + 1
                }
                for rank, i in enumerate(ranked_indices)
            ]
        }

# Test the retriever
retriever = HyDERetriever(documents=documents, doc_embeddings=doc_embeddings)

# Compare on a few queries
test_queries = [
    "What happens when packets get lost?",
    "How do computers establish a connection?",
    "Why is UDP faster?"
]

for q in test_queries:
    print(f"\nQuery: {q}")
    
    # Standard
    standard = retriever.retrieve(q, top_k=1, use_hyde=False)
    print(f"  Standard: [{standard['results'][0]['score']:.3f}] {standard['results'][0]['document'][:60]}...")
    
    # HyDE
    hyde = retriever.retrieve(q, top_k=1, use_hyde=True)
    print(f"  HyDE:     [{hyde['results'][0]['score']:.3f}] {hyde['results'][0]['document'][:60]}...")


Query: What happens when packets get lost?
  Standard: [0.584] TCP uses a sliding window protocol with acknowledgments. Whe...
  HyDE:     [0.673] TCP uses a sliding window protocol with acknowledgments. Whe...

Query: How do computers establish a connection?
  Standard: [0.446] The three-way handshake (SYN, SYN-ACK, ACK) establishes a TC...
  HyDE:     [0.541] The three-way handshake (SYN, SYN-ACK, ACK) establishes a TC...

Query: Why is UDP faster?
  Standard: [0.680] UDP is a connectionless protocol that does not guarantee del...
  HyDE:     [0.778] UDP is a connectionless protocol that does not guarantee del...


## Part 2 Parent Retrieval

You will index smaller chunks and return larger parent context.


In [5]:
# Sample document with clear hierarchy
SAMPLE_DOCUMENT = """
# Introduction to Machine Learning

Machine learning is a subset of artificial intelligence that enables systems to learn from data. 
Rather than being explicitly programmed, these systems identify patterns and make decisions with minimal human intervention.
The field has grown rapidly due to increased computational power and data availability.

## Supervised Learning

Supervised learning uses labeled training data to learn a mapping from inputs to outputs.
Common algorithms include linear regression, decision trees, and neural networks.
The model is trained on examples where the correct answer is known, then tested on unseen data.

### Classification

Classification predicts discrete categories. Examples include spam detection (spam/not spam) 
and image recognition (cat/dog/bird). Evaluation metrics include accuracy, precision, recall, and F1 score.

### Regression

Regression predicts continuous values. Examples include house price prediction and stock forecasting.
Evaluation metrics include MSE (Mean Squared Error) and R-squared.

## Unsupervised Learning

Unsupervised learning finds patterns in unlabeled data. The algorithm discovers structure without guidance.
This is useful when you don't know what patterns exist in your data.

### Clustering

Clustering groups similar items together. K-means is a popular algorithm that partitions data into K clusters.
Applications include customer segmentation and document organization.

### Dimensionality Reduction

Dimensionality reduction simplifies data by reducing the number of features.
PCA (Principal Component Analysis) projects data onto principal components that capture the most variance.
This helps with visualization and removes redundant features.
"""

print(f"Document length: {len(SAMPLE_DOCUMENT)} characters")

Document length: 1729 characters


In [6]:
import re
from typing import Optional

@dataclass
class Chunk:
    """A chunk of text with hierarchy information."""
    id: str
    text: str
    level: str  # 'document', 'section', 'subsection', 'paragraph'
    parent_id: Optional[str] = None
    children_ids: list[str] = field(default_factory=list)
    embedding: Optional[np.ndarray] = None

def create_hierarchical_chunks(document: str) -> dict[str, Chunk]:
    """Parse a markdown document into hierarchical chunks."""
    chunks = {}
    
    # Create root document chunk
    doc_id = "doc_0"
    chunks[doc_id] = Chunk(
        id=doc_id,
        text=document.strip(),
        level="document"
    )
    
    # Split by ## (sections)
    sections = re.split(r'\n## ', document)
    current_parent = doc_id
    
    for i, section in enumerate(sections[1:], 1):  # Skip the intro
        section_id = f"section_{i}"
        section_text = "## " + section.strip()
        
        chunks[section_id] = Chunk(
            id=section_id,
            text=section_text,
            level="section",
            parent_id=doc_id
        )
        chunks[doc_id].children_ids.append(section_id)
        
        # Split section by ### (subsections)
        subsections = re.split(r'\n### ', section)
        
        for j, subsection in enumerate(subsections[1:], 1):  # Skip section header
            subsection_id = f"section_{i}_sub_{j}"
            subsection_text = "### " + subsection.strip()
            
            chunks[subsection_id] = Chunk(
                id=subsection_id,
                text=subsection_text,
                level="subsection",
                parent_id=section_id
            )
            chunks[section_id].children_ids.append(subsection_id)
            
            # Split subsection into paragraphs (sentences)
            paragraphs = [p.strip() for p in subsection_text.split('\n\n') if p.strip()]
            
            for k, para in enumerate(paragraphs):
                if para.startswith('###'):
                    continue  # Skip the header
                para_id = f"section_{i}_sub_{j}_para_{k}"
                
                chunks[para_id] = Chunk(
                    id=para_id,
                    text=para,
                    level="paragraph",
                    parent_id=subsection_id
                )
                chunks[subsection_id].children_ids.append(para_id)
    
    return chunks

# Create hierarchy
chunks = create_hierarchical_chunks(SAMPLE_DOCUMENT)

# Show hierarchy
print("Document Hierarchy:")
for chunk_id, chunk in chunks.items():
    indent = {"document": 0, "section": 1, "subsection": 2, "paragraph": 3}[chunk.level]
    preview = chunk.text[:50].replace('\n', ' ') + "..."
    print(f"{'  ' * indent}[{chunk.level}] {chunk_id}: {preview}")

Document Hierarchy:
[document] doc_0: # Introduction to Machine Learning  Machine learni...
  [section] section_1: ## Supervised Learning  Supervised learning uses l...
    [subsection] section_1_sub_1: ### Classification  Classification predicts discre...
      [paragraph] section_1_sub_1_para_1: Classification predicts discrete categories. Examp...
    [subsection] section_1_sub_2: ### Regression  Regression predicts continuous val...
      [paragraph] section_1_sub_2_para_1: Regression predicts continuous values. Examples in...
  [section] section_2: ## Unsupervised Learning  Unsupervised learning fi...
    [subsection] section_2_sub_1: ### Clustering  Clustering groups similar items to...
      [paragraph] section_2_sub_1_para_1: Clustering groups similar items together. K-means ...
    [subsection] section_2_sub_2: ### Dimensionality Reduction  Dimensionality reduc...
      [paragraph] section_2_sub_2_para_1: Dimensionality reduction simplifies data by reduci...


In [7]:
@dataclass
class ParentDocumentRetriever:
    """Index small chunks, return parent documents."""
    chunks: dict[str, Chunk]
    index_level: str = "paragraph"  # Level to index (small chunks)
    return_level: str = "subsection"  # Level to return (larger context)
    
    def __post_init__(self):
        """Embed chunks at the index level."""
        self.indexed_chunks = [
            chunk for chunk in self.chunks.values() 
            if chunk.level == self.index_level
        ]
        
        print(f"Embedding {len(self.indexed_chunks)} {self.index_level}-level chunks...")
        for chunk in self.indexed_chunks:
            chunk.embedding = get_embedding(chunk.text)
    
    def _get_parent_at_level(self, chunk_id: str, target_level: str) -> Chunk:
        """Traverse up the hierarchy to find parent at target level."""
        chunk = self.chunks[chunk_id]
        
        while chunk.level != target_level and chunk.parent_id:
            chunk = self.chunks[chunk.parent_id]
        
        return chunk
    
    def retrieve(
        self, 
        query: str, 
        top_k: int = 3,
        return_parent: bool = True
    ) -> list[dict]:
        """Retrieve by matching small chunks, returning larger context."""
        
        query_embedding = get_embedding(query)
        
        # Score all indexed chunks
        scored = []
        for chunk in self.indexed_chunks:
            score = cosine_similarity(query_embedding, chunk.embedding)
            scored.append((chunk, score))
        
        # Sort by score
        scored.sort(key=lambda x: x[1], reverse=True)
        
        # Get unique parents (avoid duplicates)
        seen_parents = set()
        results = []
        
        for chunk, score in scored:
            if return_parent:
                parent = self._get_parent_at_level(chunk.id, self.return_level)
                if parent.id in seen_parents:
                    continue
                seen_parents.add(parent.id)
                
                results.append({
                    "matched_chunk": chunk.text,
                    "matched_chunk_id": chunk.id,
                    "returned_content": parent.text,
                    "returned_chunk_id": parent.id,
                    "score": score
                })
            else:
                results.append({
                    "matched_chunk": chunk.text,
                    "matched_chunk_id": chunk.id,
                    "returned_content": chunk.text,
                    "returned_chunk_id": chunk.id,
                    "score": score
                })
            
            if len(results) >= top_k:
                break
        
        return results

# Create retriever
parent_retriever = ParentDocumentRetriever(
    chunks=chunks,
    index_level="paragraph",
    return_level="subsection"
)

Embedding 4 paragraph-level chunks...


In [8]:
# Test parent document retrieval
query = "How do you measure classification performance?"

print(f"Query: {query}\n")

# Without parent retrieval (just the matched paragraph)
print("=== Standard Retrieval (return matched chunk only) ===")
results = parent_retriever.retrieve(query, top_k=1, return_parent=False)
for r in results:
    print(f"Score: {r['score']:.3f}")
    print(f"Matched: {r['matched_chunk']}")
    print(f"Returned: {r['returned_content']}")

print("\n=== Parent Document Retrieval (return subsection) ===")
results = parent_retriever.retrieve(query, top_k=1, return_parent=True)
for r in results:
    print(f"Score: {r['score']:.3f}")
    print(f"Matched paragraph: {r['matched_chunk'][:80]}...")
    print(f"\nReturned subsection:\n{r['returned_content']}")

Query: How do you measure classification performance?

=== Standard Retrieval (return matched chunk only) ===
Score: 0.576
Matched: Classification predicts discrete categories. Examples include spam detection (spam/not spam) 
and image recognition (cat/dog/bird). Evaluation metrics include accuracy, precision, recall, and F1 score.
Returned: Classification predicts discrete categories. Examples include spam detection (spam/not spam) 
and image recognition (cat/dog/bird). Evaluation metrics include accuracy, precision, recall, and F1 score.

=== Parent Document Retrieval (return subsection) ===
Score: 0.575
Matched paragraph: Classification predicts discrete categories. Examples include spam detection (sp...

Returned subsection:
### Classification

Classification predicts discrete categories. Examples include spam detection (spam/not spam) 
and image recognition (cat/dog/bird). Evaluation metrics include accuracy, precision, recall, and F1 score.


## When Parent Retrieval Helps

You should use parent retrieval when answer quality depends on surrounding context.


## Part 3 Combined Strategy

You will combine HyDE and parent retrieval in one flow.


In [9]:
@dataclass
class AdvancedRetriever:
    """Combined HyDE + Parent Document retrieval."""
    chunks: dict[str, Chunk]
    index_level: str = "paragraph"
    return_level: str = "subsection"
    indexed_chunks: list[Chunk] = field(default_factory=list)
    
    def __post_init__(self):
        """Embed chunks at the index level."""
        self.indexed_chunks = [
            chunk for chunk in self.chunks.values() 
            if chunk.level == self.index_level
        ]
        
        # Only embed if not already done
        needs_embedding = [c for c in self.indexed_chunks if c.embedding is None]
        if needs_embedding:
            print(f"Embedding {len(needs_embedding)} chunks...")
            for chunk in needs_embedding:
                chunk.embedding = get_embedding(chunk.text)
    
    def _get_parent_at_level(self, chunk_id: str, target_level: str) -> Chunk:
        chunk = self.chunks[chunk_id]
        while chunk.level != target_level and chunk.parent_id:
            chunk = self.chunks[chunk.parent_id]
        return chunk
    
    def retrieve(
        self,
        query: str,
        top_k: int = 3,
        use_hyde: bool = False,
        return_parent: bool = True
    ) -> dict:
        """Retrieve with optional HyDE and parent document return."""
        
        # Step 1: Get query embedding (with or without HyDE)
        hypothetical = None
        if use_hyde:
            hypothetical = generate_hypothetical_answer(query)
            query_embedding = get_embedding(hypothetical)
        else:
            query_embedding = get_embedding(query)
        
        # Step 2: Score chunks
        scored = []
        for chunk in self.indexed_chunks:
            score = cosine_similarity(query_embedding, chunk.embedding)
            scored.append((chunk, score))
        scored.sort(key=lambda x: x[1], reverse=True)
        
        # Step 3: Collect results (with optional parent expansion)
        seen_parents = set()
        results = []
        
        for chunk, score in scored:
            if return_parent:
                parent = self._get_parent_at_level(chunk.id, self.return_level)
                if parent.id in seen_parents:
                    continue
                seen_parents.add(parent.id)
                returned_content = parent.text
                returned_id = parent.id
            else:
                returned_content = chunk.text
                returned_id = chunk.id
            
            results.append({
                "matched_chunk": chunk.text,
                "returned_content": returned_content,
                "returned_id": returned_id,
                "score": score
            })
            
            if len(results) >= top_k:
                break
        
        return {
            "query": query,
            "hypothetical": hypothetical,
            "use_hyde": use_hyde,
            "return_parent": return_parent,
            "results": results
        }

# Create combined retriever (reuses existing embeddings)
advanced_retriever = AdvancedRetriever(
    chunks=chunks,
    index_level="paragraph",
    return_level="subsection"
)

In [10]:
# Compare all four combinations
query = "What algorithm groups similar things together?"

print(f"Query: {query}\n")
print("=" * 60)

configs = [
    {"use_hyde": False, "return_parent": False, "name": "Standard (no HyDE, no parent)"},
    {"use_hyde": True, "return_parent": False, "name": "HyDE only"},
    {"use_hyde": False, "return_parent": True, "name": "Parent Doc only"},
    {"use_hyde": True, "return_parent": True, "name": "HyDE + Parent Doc"},
]

for config in configs:
    print(f"\n### {config['name']} ###")
    result = advanced_retriever.retrieve(
        query, 
        top_k=1, 
        use_hyde=config["use_hyde"],
        return_parent=config["return_parent"]
    )
    
    if result["hypothetical"]:
        print(f"Hypothetical: {result['hypothetical'][:100]}...")
    
    top = result["results"][0]
    print(f"Score: {top['score']:.3f}")
    print(f"Matched: {top['matched_chunk'][:80]}...")
    print(f"Returned ({len(top['returned_content'])} chars): {top['returned_content'][:100]}...")

Query: What algorithm groups similar things together?


### Standard (no HyDE, no parent) ###
Score: 0.561
Matched: Clustering groups similar items together. K-means is a popular algorithm that pa...
Returned (180 chars): Clustering groups similar items together. K-means is a popular algorithm that partitions data into K...

### HyDE only ###
Hypothetical: Clustering algorithms are unsupervised learning methods that automatically group similar items toget...
Score: 0.709
Matched: Clustering groups similar items together. K-means is a popular algorithm that pa...
Returned (180 chars): Clustering groups similar items together. K-means is a popular algorithm that partitions data into K...

### Parent Doc only ###
Score: 0.561
Matched: Clustering groups similar items together. K-means is a popular algorithm that pa...
Returned (196 chars): ### Clustering

Clustering groups similar items together. K-means is a popular algorithm that partit...

### HyDE + Parent Doc ###
Hypothetical: Cluster

## Part 4 RAG Integration

You will connect advanced retrieval to end to end answer generation.


In [11]:
def rag_with_advanced_retrieval(
    question: str,
    retriever: AdvancedRetriever,
    use_hyde: bool = True,
    return_parent: bool = True,
    top_k: int = 2
) -> dict:
    """Complete RAG pipeline with advanced retrieval."""
    
    # Step 1: Retrieve
    retrieval = retriever.retrieve(
        question,
        top_k=top_k,
        use_hyde=use_hyde,
        return_parent=return_parent
    )
    
    # Step 2: Format context
    context = "\n\n---\n\n".join([
        r["returned_content"] for r in retrieval["results"]
    ])
    
    # Step 3: Generate answer
    response = client.responses.create(
        model=MODEL,
        instructions="""Answer the question based on the provided context.
If the context doesn't contain enough information, say so.
Be concise and accurate.""",
        input=f"""Context:
{context}

Question: {question}

Answer:"""
    )
    
    return {
        "question": question,
        "answer": response.output_text,
        "retrieval": retrieval,
        "context_length": len(context)
    }

# Test RAG pipeline
questions = [
    "What's the difference between classification and regression?",
    "How do you reduce the number of features in a dataset?",
    "What is K-means used for?"
]

for q in questions:
    print(f"\nQ: {q}")
    result = rag_with_advanced_retrieval(q, advanced_retriever)
    print(f"A: {result['answer']}")
    print(f"   (Retrieved {len(result['retrieval']['results'])} chunks, {result['context_length']} chars)")


Q: What's the difference between classification and regression?
A: Classification predicts discrete categories (e.g., spam vs. not spam, cat/dog/bird); common metrics: accuracy, precision, recall, F1.  
Regression predicts continuous numeric values (e.g., house prices, stock forecasts); common metrics: MSE, R-squared.
   (Retrieved 2 chunks, 412 chars)

Q: How do you reduce the number of features in a dataset?
A: Use dimensionality reduction. For example, apply PCA (Principal Component Analysis) to project the data onto a smaller number of principal components that capture most of the variance—keeping only the top components reduces features and removes redundant information (also useful for visualization).
   (Retrieved 2 chunks, 503 chars)

Q: What is K-means used for?
A: K-means is used for clustering — it partitions data into K groups of similar items (e.g., for customer segmentation or document organization).
   (Retrieved 2 chunks, 424 chars)


## Practical Guidelines

You will choose method combinations based on quality and latency needs.


## Exercises With Starter and Solution

### Exercise 1

You will implement multi HyDE with averaged hypothetical embeddings.


In [12]:
# Starter code for Exercise 1
def multi_hyde_retrieve(
    query: str,
    retriever: AdvancedRetriever,
    n_hypotheticals: int = 3,
    top_k: int = 3
) -> list[dict]:
    # TODO generate multiple hypotheticals and average their embeddings
    pass


In [13]:
# Solution for Exercise 1
def multi_hyde_retrieve(
    query: str,
    retriever: AdvancedRetriever,
    n_hypotheticals: int = 3,
    top_k: int = 3
) -> dict:
    hypotheticals = [generate_hypothetical_answer(query) for _ in range(n_hypotheticals)]
    hypo_embeddings = [get_embedding(h) for h in hypotheticals]
    avg_embedding = np.mean(np.vstack(hypo_embeddings), axis=0)

    scored = []
    for chunk in retriever.indexed_chunks:
        score = cosine_similarity(avg_embedding, chunk.embedding)
        parent = retriever._get_parent_at_level(chunk.id, retriever.return_level)
        scored.append((chunk, parent, score))

    scored.sort(key=lambda x: x[2], reverse=True)

    results = []
    seen_parent_ids = set()
    for chunk, parent, score in scored:
        if parent.id in seen_parent_ids:
            continue
        seen_parent_ids.add(parent.id)
        results.append({
            'matched_chunk': chunk.text,
            'returned_content': parent.text,
            'returned_id': parent.id,
            'score': float(score),
        })
        if len(results) >= top_k:
            break

    return {
        'query': query,
        'hypotheticals': hypotheticals,
        'results': results,
    }

example = multi_hyde_retrieve('How does clustering work?', advanced_retriever, n_hypotheticals=2, top_k=2)
print(example['query'])
for row in example['results']:
    print(round(row['score'], 3), row['returned_id'])


How does clustering work?
0.678 section_2_sub_1
0.385 section_1_sub_1


### Exercise 2

You will choose parent level from query complexity.


In [14]:
# Starter code for Exercise 2
def adaptive_parent_retrieve(
    query: str,
    retriever: AdvancedRetriever,
    top_k: int = 3
) -> list[dict]:
    # TODO choose parent level dynamically from query complexity
    pass


In [15]:
# Solution for Exercise 2
def adaptive_parent_retrieve(
    query: str,
    retriever: AdvancedRetriever,
    top_k: int = 3
) -> dict:
    token_count = len(query.split())

    if token_count <= 4:
        target_level = 'paragraph'
    elif token_count <= 8:
        target_level = 'subsection'
    else:
        target_level = 'section'

    query_embedding = get_embedding(query)
    scored = []
    for chunk in retriever.indexed_chunks:
        score = cosine_similarity(query_embedding, chunk.embedding)
        scored.append((chunk, score))
    scored.sort(key=lambda x: x[1], reverse=True)

    results = []
    seen = set()
    for chunk, score in scored:
        if target_level == 'paragraph':
            returned = chunk
        else:
            returned = retriever._get_parent_at_level(chunk.id, target_level)

        if returned.id in seen:
            continue
        seen.add(returned.id)

        results.append({
            'matched_chunk': chunk.text,
            'returned_content': returned.text,
            'returned_id': returned.id,
            'score': float(score),
            'target_level': target_level,
        })

        if len(results) >= top_k:
            break

    return {'query': query, 'target_level': target_level, 'results': results}

adaptive = adaptive_parent_retrieve('How to evaluate classification models in practice', advanced_retriever, top_k=1)
print(adaptive['target_level'])


subsection


### Exercise 3

You will create domain aware HyDE prompting.


In [16]:
# Starter code for Exercise 3
def domain_aware_hyde(
    query: str,
    domain: str,
) -> str:
    # TODO generate a domain aware hypothetical answer
    pass


In [ ]:
# Solution for Exercise 3
def domain_aware_hyde(
    query: str,
    domain: str,
) -> str:
    guidance = {
        'technical': 'Use protocol and systems vocabulary.',
        'legal': 'Use legal terms and obligation language.',
        'medical': 'Use clinical terms and cautious wording.',
        'general': 'Use clear educational wording.',
    }
    domain_instruction = guidance.get(domain, guidance['general'])

    response = client.responses.create(
        model=MODEL,
        instructions=(
            'Write a short hypothetical answer passage that could match real documents. '
            + domain_instruction
        ),
        input=query,
    )
    return response.output_text

for d in ['technical', 'medical', 'general']:
    out = domain_aware_hyde('How does packet retransmission work?', d)
    print(d, ':', out[:80], '...')


## What You Learned

You implemented HyDE, parent retrieval, and combined retrieval for RAG.
